In [1]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

2026-01-04 08:45:13.933048: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-04 08:45:15.316530: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


cuda


In [2]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH
).to(device)
model.eval()

print(model)

model.load_state_dict(torch.load("best_metric_model.pth"))

UNet(
  (model): Sequential(
    (0): ResidualUnit(
      (conv): Sequential(
        (unit0): Convolution(
          (conv): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
        (unit1): Convolution(
          (conv): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
      )
      (residual): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    )
    (1): SkipConnection(
      (submodule): Sequential(
        (0): ResidualUnit(
          (conv): Sequential(


<All keys matched successfully>

In [3]:
model.model[1]

SkipConnection(
  (submodule): Sequential(
    (0): ResidualUnit(
      (conv): Sequential(
        (unit0): Convolution(
          (conv): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
        (unit1): Convolution(
          (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
      )
      (residual): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    )
    (1): SkipConnection(
      (submodule): Sequential(
        (0): ResidualUnit(
          (con

In [4]:
import glob
from pathlib import Path
from monai.data import Dataset, DataLoader
from monai.data.image_reader import NibabelReader
from sklearn.model_selection import train_test_split

try:
    import nibabel  # ensure dependency available
except ImportError as exc:
    raise ImportError("nibabel is required for loading NIfTI files. Install via `pip install nibabel`." ) from exc
import numpy as np

# Initialize NIfTI reader
nibabel_reader = NibabelReader()

# Data directory
data_dir = Path("data/ATLAS_2/Training")

# Find all image and mask pairs
image_files = sorted(glob.glob(str(data_dir / "**/*_T1w.nii.gz"), recursive=True))
mask_files = sorted(glob.glob(str(data_dir / "**/*_label-L_desc-T1lesion_mask.nii.gz"), recursive=True))

print(f"Found {len(image_files)} image files")
print(f"Found {len(mask_files)} mask files")

# Create data dictionary
indistribution_data = [
    {"image": img, "label": mask}
    for img, mask in zip(image_files, mask_files)
]

# Split into train and validation (80/20)
train_data, val_data = train_test_split(indistribution_data, test_size=0.2, random_state=42)
print(f"Train data: {len(train_data)} files")
print(f"Validation data: {len(val_data)} files")

Found 655 image files
Found 655 mask files
Train data: 524 files
Validation data: 131 files


In [5]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    ScaleIntensityRanged,
    CropForegroundd,
    ToTensord,
    Resized
)
from monai.data import Dataset, DataLoader

transform = Compose([
    LoadImaged(keys=["image", "label"], reader=nibabel_reader, image_only=False),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=1000, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    Resized(keys=["image", "label"], spatial_size=(96, 96, 96), mode=("trilinear", "nearest")),
    ToTensord(keys=["image", "label"]),
])

train_id_ds = Dataset(data=train_data, transform=transform)
train_id_loader = DataLoader(train_id_ds, batch_size=1, num_workers=4, pin_memory=True)

val_id_ds = Dataset(data=val_data, transform=transform)
val_id_loader = DataLoader(val_id_ds, batch_size=1, num_workers=4, pin_memory=True)

monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [6]:
batch = next(iter(train_id_loader))
image = batch["image"].to(device)
label = batch["label"].to(device)
print(f"Image shape: {image.shape}, Label shape: {label.shape}")


Image shape: torch.Size([1, 1, 96, 96, 96]), Label shape: torch.Size([1, 1, 96, 96, 96])


In [7]:
import torch.nn.functional as F

def get_penultimate_features(model, image):
    with torch.no_grad():
        features = model.model[0](image)
        features = model.model[1](features)
        return features

features = get_penultimate_features(model, image)
features.shape

torch.Size([1, 32, 48, 48, 48])

In [ ]:
import torch.nn.functional as F

CLASSES = [0, 1]

class GaussianMeanCalculator:
    def __init__(self):
        self.class_accumulator = {c: torch.zeros(32, dtype=torch.float64).to(device) for c in CLASSES}
        self.class_counts = {c: 0 for c in CLASSES}
    
    def update(self, image, label):
        features = get_penultimate_features(model, image)
        label_interpolated = F.interpolate(label, size=(features.shape[2:5]), mode='nearest').long()

        features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
        labels_flat = label_interpolated.reshape(-1)

        for c in CLASSES:
            self.class_accumulator[c] += features_flat[labels_flat == c].sum(dim=0)
            self.class_counts[c] += (labels_flat == c).sum().item()
    
    def compute_means(self):
        means = {c: self.class_accumulator[c] / self.class_counts[c] for c in CLASSES}
        return means

In [9]:
from tqdm import tqdm

def calculate_global_class_means(loader):
    calculator = GaussianMeanCalculator()

    for batch in tqdm(loader):
        image = batch["image"].to(device)
        label = batch["label"].to(device)
        calculator.update(image, label)

    means = calculator.compute_means()
    return means

class_means = calculate_global_class_means(train_id_loader)
print(class_means)

100%|██████████| 524/524 [01:00<00:00,  8.60it/s]

{0: metatensor([ 0.2534,  0.3448,  0.2361, -0.0391,  0.3396,  0.3369,  0.3215,  0.1538,
         0.2461,  0.0892,  0.2018,  0.1046,  0.0359,  0.1766,  0.4020,  0.1486,
         0.7612,  0.7707,  0.7182,  0.8149,  0.7270,  0.6393,  0.8275,  0.7669,
         0.7893,  0.6155,  0.5848,  0.6921,  0.6967,  0.7505,  0.7239,  0.7929],
       device='cuda:0'), 1: metatensor([ 0.1293,  0.4316,  0.2679, -0.0813,  0.2017,  0.3732,  0.2016,  0.1126,
         0.2958, -0.0702,  0.1241,  0.1214,  0.0389,  0.0483,  0.2066,  0.2524,
         0.7394,  0.3597,  1.1786,  0.7220,  0.5374,  1.0417,  3.2448,  1.5677,
         1.5153,  0.7131,  0.3520,  1.1188,  1.5866,  1.6623,  1.4151,  0.6083],
       device='cuda:0')}


In [ ]:
class GaussianTiedCovarianceCalculator:
    def __init__(self, class_means):
        self.class_means = class_means
        self.cov_accumulator = torch.zeros(32, 32, dtype=torch.float64).to(device)
        self.total_count = 0
    
    def update(self, image, label):
        features = get_penultimate_features(model, image)
        label_interpolated = F.interpolate(label, size=(features.shape[2:5]), mode='nearest').long()

        features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
        labels_flat = label_interpolated.reshape(-1)

        for c in CLASSES:
            class_features = features_flat[labels_flat == c]
            mean = self.class_means[c].unsqueeze(0)
            diffs = class_features - mean
            self.cov_accumulator += diffs.t() @ diffs
            self.total_count += class_features.size(0)
    
    def compute_covariance(self):
        covariance = self.cov_accumulator / self.total_count
        return covariance

In [11]:
def calculate_global_tied_covariance(loader, class_means):
    calculator = GaussianTiedCovarianceCalculator(class_means)

    for batch in tqdm(loader):
        image = batch["image"].to(device)
        label = batch["label"].to(device)
        calculator.update(image, label)

    covariance = calculator.compute_covariance()
    return covariance

covariance = calculate_global_tied_covariance(train_id_loader, class_means)
print(covariance)

100%|██████████| 524/524 [01:02<00:00,  8.44it/s]

metatensor([[ 0.1020, -0.0828, -0.0678,  ..., -0.0055, -0.0069, -0.0292],
        [-0.0828,  0.4042,  0.1562,  ..., -0.0028,  0.0066,  0.0436],
        [-0.0678,  0.1562,  0.4162,  ...,  0.0019,  0.0081,  0.0287],
        ...,
        [-0.0055, -0.0028,  0.0019,  ...,  1.3503, -0.2251, -0.2490],
        [-0.0069,  0.0066,  0.0081,  ..., -0.2251,  1.4266,  0.5695],
        [-0.0292,  0.0436,  0.0287,  ..., -0.2490,  0.5695,  0.6792]],
       device='cuda:0')


In [ ]:
def calculate_mahalanobis_confidence_score(features, class_means, covariance):
    features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
    inv_cov = torch.linalg.pinv(covariance + 1e-6 * torch.eye(covariance.size(0)).to(device))
    scores = []
    for c in CLASSES:
        mean = class_means[c]
        diffs = features_flat - mean
        tmp = diffs @ inv_cov
        score = torch.sum(tmp * diffs, dim=1)
        scores.append(-score)
    scores = torch.stack(scores, dim=0)
    max_scores = torch.max(scores, dim=0).values
    return max_scores.reshape(features.shape[2:])

batch = next(iter(val_id_loader))
image = batch["image"].to(device)
features = get_penultimate_features(model, image)

score = calculate_mahalanobis_confidence_score(features, class_means, covariance)
print(score.shape)


W20260104 08:47:22.236747 140673988770752 Context.cpp:456] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())


torch.Size([48, 48, 48])


In [13]:
score.mean()

metatensor(-35.4019, device='cuda:0')

In [16]:
scores = []
for batch in tqdm(val_id_loader):
    image = batch["image"].to(device)
    features = get_penultimate_features(model, image)
    score = calculate_mahalanobis_confidence_score(features, class_means, covariance)
    scores.append(score.mean().item())
print(sum(scores) / len(scores))

100%|██████████| 131/131 [00:16<00:00,  8.16it/s]

-31.65856776346687


In [18]:
scores = []
for batch in tqdm(train_id_loader):
    image = batch["image"].to(device)
    features = get_penultimate_features(model, image)
    score = calculate_mahalanobis_confidence_score(features, class_means, covariance)
    scores.append(score.mean().item())
print(sum(scores) / len(scores))

100%|██████████| 524/524 [01:00<00:00,  8.68it/s]

-31.471530146271217


In [26]:
# CHAOS Data Loader
# CHAOS dataset contains CT and MR data in DICOM format
# For OOD detection, we'll use all available CHAOS data (Train + Test, CT + MR)

from monai.data.image_reader import ITKReader
from pathlib import Path

# Initialize DICOM reader (ITKReader can handle DICOM series)
dicom_reader = ITKReader()

# Base CHAOS data directory
chaos_base_dir = Path("data/chaos")

# Function to collect CT data from a directory
def collect_ct_data(ct_dir):
    """Collect CT DICOM series from a CT directory."""
    ct_data = []
    if not ct_dir.exists():
        return ct_data
    
    patient_dirs = sorted([d for d in ct_dir.iterdir() if d.is_dir() and d.name.isdigit()])
    for patient_dir in patient_dirs:
        dicom_dir = patient_dir / "DICOM_anon"
        if dicom_dir.exists():
            dicom_files = sorted(list(dicom_dir.glob("*.dcm")))
            if len(dicom_files) > 0:
                ct_data.append({"image": str(dicom_dir), "modality": "CT"})
    return ct_data

# Function to collect MR data from a directory
def collect_mr_data(mr_dir):
    """Collect MR DICOM series from a MR directory (T1DUAL and T2SPIR)."""
    mr_data = []
    if not mr_dir.exists():
        return mr_data
    
    patient_dirs = sorted([d for d in mr_dir.iterdir() if d.is_dir() and d.name.isdigit()])
    for patient_dir in patient_dirs:
        # CHAOS MR has T1DUAL and T2SPIR sequences
        for sequence in ["T1DUAL", "T2SPIR"]:
            seq_dir = patient_dir / sequence
            if seq_dir.exists():
                dicom_files = sorted(list(seq_dir.glob("*.dcm")))
                if len(dicom_files) > 0:
                    mr_data.append({
                        "image": str(seq_dir), 
                        "modality": "MR",
                        "sequence": sequence
                    })
    return mr_data

# Collect all CHAOS data
chaos_data = []

# Train CT
train_ct_dir = chaos_base_dir / "CHAOS_Train_Sets" / "Train_Sets" / "CT"
train_ct_data = collect_ct_data(train_ct_dir)
chaos_data.extend(train_ct_data)
print(f"Found {len(train_ct_data)} Train CT volumes")

# Test CT
test_ct_dir = chaos_base_dir / "CHAOS_Test_Sets" / "Test_Sets" / "CT"
test_ct_data = collect_ct_data(test_ct_dir)
chaos_data.extend(test_ct_data)
print(f"Found {len(test_ct_data)} Test CT volumes")

# Train MR
train_mr_dir = chaos_base_dir / "CHAOS_Train_Sets" / "Train_Sets" / "MR"
train_mr_data = collect_mr_data(train_mr_dir)
chaos_data.extend(train_mr_data)
print(f"Found {len(train_mr_data)} Train MR volumes")

# Test MR
test_mr_dir = chaos_base_dir / "CHAOS_Test_Sets" / "Test_Sets" / "MR"
test_mr_data = collect_mr_data(test_mr_dir)
chaos_data.extend(test_mr_data)
print(f"Found {len(test_mr_data)} Test MR volumes")

print(f"Total CHAOS volumes: {len(chaos_data)}")

# CHAOS transforms - similar to ATLAS but adapted for different modalities
# Note: CHAOS data doesn't have labels for OOD detection, so we only load images
chaos_transform = Compose([
    LoadImaged(keys=["image"], reader=dicom_reader, image_only=False),
    EnsureChannelFirstd(keys=["image"]),
    Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
    Orientationd(keys=["image"], axcodes="RAS"),
    # CT data typically has HU values (-1000 to 1000)
    # MR data has different intensity ranges, but we'll normalize similarly
    # For OOD detection, we want to normalize to [0, 1] range
    ScaleIntensityRanged(keys=["image"], a_min=-1000, a_max=1000, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image"], source_key="image"),
    Resized(keys=["image"], spatial_size=(96, 96, 96), mode="trilinear"),
    ToTensord(keys=["image"]),
])

chaos_ds = Dataset(data=chaos_data, transform=chaos_transform)
chaos_loader = DataLoader(chaos_ds, batch_size=1, num_workers=4, pin_memory=True)

print(f"CHAOS dataloader created with {len(chaos_ds)} samples")


Found 20 Train CT volumes
Found 20 Test CT volumes
Found 0 Train MR volumes
Found 0 Test MR volumes
Total CHAOS volumes: 40
CHAOS dataloader created with 40 samples


In [27]:
scores = []
for batch in tqdm(chaos_loader):
    image = batch["image"].to(device)
    features = get_penultimate_features(model, image)
    score = calculate_mahalanobis_confidence_score(features, class_means, covariance)
    scores.append(score.mean().item())
print(sum(scores) / len(scores))

100%|██████████| 40/40 [00:45<00:00,  1.14s/it]

-1526.5065460205078
